In [1]:
from samap.mapping import SAMAP
from samap.analysis import (get_mapping_scores, GenePairFinder, transfer_annotations,
                            sankey_plot, chord_plot, CellTypeTriangles, 
                            ParalogSubstitutions, FunctionalEnrichment,
                            convert_eggnog_to_homologs, GeneTriangles)
from samalg import SAM
import pandas as pd
from Bio import SeqIO
from samap.utils import (save_samap, load_samap)
import scanpy as sc
import matplotlib.colors
import matplotlib.pyplot as plt
import numpy as np
from scipy import stats
from scipy import sparse 
from scipy import cluster
import seaborn as sns
import random
import sklearn
from scipy.stats import poisson
from sklearn.neighbors import KernelDensity
import time
import dill
from scipy.optimize import minimize
import pickle
import itertools

/scratch/miniconda/lib/python3.7/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def orthogroup_dicts(o, level):
    org_dict_v = {}
    for item in o.index:
        v = o.loc[item,'eggNOG_OGs'].split('|' + level)[0].split(',')[-1]        
        if '|' not in v:
            if v in org_dict_v.keys():
                org_dict_v[v].append(item)
            else:
                org_dict_v[v] = [item]
    return(org_dict_v)

In [4]:
orthogroups_xt = pd.read_csv('../../BLASTMAPPING/emapper_v2/XT_emapper/XT.emapper.annotations', delimiter = '\t',skiprows = 4,skipfooter=3, index_col = '#query',engine='python')
orthogroups_cj = pd.read_csv('../../BLASTMAPPING/emapper_v2/CJ_emapper/CJ.emapper.annotations', delimiter = '\t',skiprows = 4,skipfooter=3, index_col = '#query',engine='python')
orthogroups_dr = pd.read_csv('../../BLASTMAPPING/emapper_v2/DR_emapper/DR.emapper.annotations', delimiter = '\t',skiprows = 4,skipfooter=3, index_col = '#query',engine='python')
orthogroups_mo = pd.read_csv('../../BLASTMAPPING/emapper_v2/MO_emapper/MO.emapper.annotations', delimiter = '\t',skiprows = 4,skipfooter=3, index_col = '#query',engine='python')
orthogroups_ac = pd.read_csv('../../BLASTMAPPING/emapper_v2/AC_emapper/AC.emapper.annotations', delimiter = '\t',skiprows = 4,skipfooter=3, index_col = '#query',engine='python')
orthogroups_mm = pd.read_csv('../../BLASTMAPPING/emapper_v2/MM_emapper/MM.emapper.annotations', delimiter = '\t',skiprows = 4,skipfooter=3, index_col = '#query',engine='python')

In [5]:
org_dict_vert_mm= orthogroup_dicts(orthogroups_mm, 'Vertebrata')
org_dict_vert_mo= orthogroup_dicts(orthogroups_mo, 'Vertebrata')
org_dict_vert_cj= orthogroup_dicts(orthogroups_cj, 'Vertebrata')
org_dict_vert_ac= orthogroup_dicts(orthogroups_ac, 'Vertebrata')
org_dict_vert_xt= orthogroup_dicts(orthogroups_xt, 'Vertebrata')
org_dict_vert_dr= orthogroup_dicts(orthogroups_dr, 'Vertebrata')

In [6]:
org_dict_chord_mm= orthogroup_dicts(orthogroups_mm, 'Chordata')
org_dict_chord_mo= orthogroup_dicts(orthogroups_mo, 'Chordata')
org_dict_chord_cj= orthogroup_dicts(orthogroups_cj, 'Chordata')
org_dict_chord_ac= orthogroup_dicts(orthogroups_ac, 'Chordata')
org_dict_chord_xt= orthogroup_dicts(orthogroups_xt, 'Chordata')
org_dict_chord_dr= orthogroup_dicts(orthogroups_dr, 'Chordata')

In [7]:
org_dict_bil_mm= orthogroup_dicts(orthogroups_mm, 'Bilateria')
org_dict_bil_mo= orthogroup_dicts(orthogroups_mo, 'Bilateria')
org_dict_bil_cj= orthogroup_dicts(orthogroups_cj, 'Bilateria')
org_dict_bil_ac= orthogroup_dicts(orthogroups_ac, 'Bilateria')
org_dict_bil_xt= orthogroup_dicts(orthogroups_xt, 'Bilateria')
org_dict_bil_dr= orthogroup_dicts(orthogroups_dr, 'Bilateria')

In [8]:
all_vert = set(org_dict_vert_mm.keys()) | set(org_dict_vert_mo.keys()) | set(org_dict_vert_cj.keys()) | set(org_dict_vert_ac.keys()) | set(org_dict_vert_xt.keys()) | set(org_dict_vert_dr.keys()) 

In [9]:
len(all_vert)

22550

In [10]:
vert_df = pd.DataFrame(index = all_vert, columns = ['MM','MO','CJ','AC','XT','DR'])

In [12]:
for item in org_dict_vert_mm.keys():
    vert_df.loc[item, 'MM'] = ','.join(org_dict_vert_mm[item])
    
for item in org_dict_vert_mo.keys():
    vert_df.loc[item, 'MO'] = ','.join(org_dict_vert_mo[item])
    
for item in org_dict_vert_cj.keys():
    vert_df.loc[item, 'CJ'] = ','.join(org_dict_vert_cj[item])
    
for item in org_dict_vert_ac.keys():
    vert_df.loc[item, 'AC'] = ','.join(org_dict_vert_ac[item])
    
for item in org_dict_vert_xt.keys():
    vert_df.loc[item, 'XT'] = ','.join(org_dict_vert_xt[item])
    
for item in org_dict_vert_dr.keys():
    vert_df.loc[item, 'DR'] = ','.join(org_dict_vert_dr[item])

In [13]:
# Write as TSV, not CSV. Cells hold comma-joined gene lists, so a comma-delimited
# file forces pandas to quote them, and any edge case there can corrupt rows and
# balloon the column count (the old *.csv had 1024 columns from 2 bad rows).
# Gene IDs never contain tabs (tab is emapper's own column delimiter) or newlines,
# so tab-separated output is collision-free.
out_path = 'Vert_emapper_allorgs_08012026.tsv'
vert_df.to_csv(out_path, sep='\t')

# Round-trip check: re-read and confirm the shape matches what we wrote, so a
# future malformed row fails loudly here instead of silently bloating the file.
_check = pd.read_csv(out_path, sep='\t', index_col=0, low_memory=False)
assert _check.shape == vert_df.shape, f"round-trip mismatch: wrote {vert_df.shape}, read {_check.shape}"
assert list(_check.columns) == list(vert_df.columns), f"unexpected columns: {list(_check.columns)}"
print(f"OK: wrote {out_path} with shape {_check.shape}, columns {list(_check.columns)}")

OK: wrote Vert_emapper_allorgs_08012026.tsv with shape (22550, 6), columns ['MM', 'MO', 'CJ', 'AC', 'XT', 'DR']
